In [1]:
import duckdb
import pandas as pd

In [2]:
# ler csv com duckdb e transformar em tabela SQL

dataset = r'dados\base_teste_tecnico.csv'

con = duckdb.connect()

con.execute(f"""
CREATE TABLE vendas AS
SELECT *
FROM read_csv_auto('{dataset}')
""")

In [3]:
# checar nomes das colunas e se há imperfeições visíveis

con.execute("""
SELECT *
FROM vendas
LIMIT 10
""").df()

,ano_mes_dia,loja,id_venda,id_cliente,faturamento
0,20250414,1,3D2C8C2AD0D66433E82A58FB5EC9DEDC,FC7E35E3C5B7B1F082F217FD984A5819,44.70
1,20250325,2,D07BB3CA6DC65BE3C812BC212040A8E8,7A8DAD40D0F099768A72FBDC91B92686,31.33
2,20250731,2,0C5E015645B1B7C8D20D6444EE4E6490,7F1F980AA088E2A428874D0C1A277B1A,191.23
3,20260117,1,B80CB58AE9463020AEE2356FD034E3D9,ACCE1F2B53C0862129C45DCE5FD79B04,57.15
4,20250131,1,A3C6EA6F59DF644A6CB74236E28D670B,96CB9CC81AC90D2DF67ED60257131281,195.86
5,20250504,2,3B7260A4AD48FF58472D7010B3FFB7DA,4AFF46B0ECBC19E4EFC67EC5E95E6C9E,22.46
6,20251113,1,C188E80C774611EBFFC6105947DC9F3C,B61B4B8B39819A6F8377612D25D9FFB2,54.74
7,20251108,2,A6B588EA558A74E94ADB483C1E5621F6,4140BC8DFA0BDB8C12E1F8E019ADB486,387.97
8,20260112,1,37A07FDC943D88998C028B202414CE3F,50142C84FA5D54CECF4D16C7AB4E9D23,60.89
9,20250128,2,A3E6C5EBD4547067E22A7960A271C7EC,8A733341A12B74B581D2DF78E729F689,7.88


In [4]:
# quantas linhas existem na base?

con.execute("""
SELECT COUNT(*) AS qtd_linhas
FROM vendas
""").df()

,qtd_linhas
0,309178


In [5]:
# quantidade de clientes únicos

con.execute("""
SELECT COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM vendas
""").df()

,clientes_unicos
0,11684


In [6]:
# quantidade de lojas únicas

con.execute("""
SELECT COUNT(DISTINCT loja) AS lojas_unicas
FROM vendas
""").df()

,lojas_unicas
0,2


In [7]:
# data mais antiga e mais recente

con.execute("""
SELECT
    MIN(ano_mes_dia) AS data_mais_antiga,
    MAX(ano_mes_dia) AS data_mais_recente
FROM vendas
""").df()

,data_mais_antiga,data_mais_recente
0,20250102,20260531


In [8]:
#

con.execute("""
SELECT
    MIN(faturamento) AS faturamento_minimo,
    MAX(faturamento) AS faturamento_maximo,
    AVG(faturamento) AS faturamento_medio,
    MEDIAN(faturamento) AS mediana_faturamento
FROM vendas
""").df()

,faturamento_minimo,faturamento_maximo,faturamento_medio,mediana_faturamento
0,-1151.03,7000.0,78.325846,48.87


In [9]:
# há valores negativos de faturamento, o que pode significar erro ou alguma devolução de valor.
# quantos registros de faturamento negativo existem na base?

con.execute("""
SELECT
    COUNT(*) AS qtd_negativos,
    ROUND(
        100 * COUNT(*) / (SELECT COUNT(*) FROM vendas),
        2
    ) AS pencentual_negativos
FROM vendas
WHERE faturamento < 0
""").df()

,qtd_negativos,pencentual_negativos
0,286,0.09


In [10]:
# quantos clientes tem valores negativos de faturamento?

con.execute("""
SELECT COUNT(DISTINCT id_cliente) AS qtd_clientes_afetados
FROM vendas
WHERE faturamento < 0
""").df()

,qtd_clientes_afetados
0,252


In [11]:
# de qual quantia negativa estamos falando?

con.execute("""
SELECT
    SUM(faturamento) AS soma_negativos,
    AVG (faturamento) AS media_negativos,
    MIN(faturamento) AS valor_minimo
FROM vendas
WHERE faturamento < 0
""").df()

,soma_negativos,media_negativos,valor_minimo
0,-23124.88,-80.856224,-1151.03


In [12]:
# além dos negativos, existem linhas zeradas de faturamento?

con.execute("""
SELECT COUNT(*) AS qtd_zerados
FROM vendas
WHERE faturamento = 0
""").df()

,qtd_zerados
0,0


In [13]:
#

con.execute("""
SELECT
    APPROX_QUANTILE(faturamento, 0.25) AS q1,
    APPROX_QUANTILE(faturamento, 0.50) AS mediana,
    APPROX_QUANTILE(faturamento, 0.75) AS q3,
    APPROX_QUANTILE(faturamento, 0.95) AS p95,
    APPROX_QUANTILE(faturamento, 0.99) AS p99,
FROM vendas
""").df()

,q1,mediana,q3,p95,p99
0,25.22095,48.87138,93.872936,242.109571,479.813825


In [14]:
# De um total de 309178 registros, temos apenas 286 negativos, o que corresponde a 0.09% da base.
# Do total de 11684 clientes únicos, 252 apresentam valores negativos de faturamento, equivalendo a 2.2%

# Esses registros negativos provavelmente correspondem a devoluções ou cancelamentos.
# Portanto, dado o baixo impacto, vou desconsiderar eles da análise.


# Sobre os quantis, vemos que 99% das compras estão abaixo de R$ 480, porém o faturamento máximo é R$ 7000.
# Ou seja, há poucos registros extremamente elevados ao comparar com o resto da base.

In [19]:
# De 20250102 a 20260531, em quantos dias houve registro de compras?

con.execute("""
SELECT
    COUNT(DISTINCT ano_mes_dia) AS dias_com_compra
FROM vendas
""").df()

,dias_com_compra
0,504


In [25]:
# estatisticas dos dias de compra

con.execute("""
SELECT
    AVG(freq) AS media_dos_dias_de_compra,
    MEDIAN(freq) AS mediana_dos_dias_de_compra,
    MAX(freq) AS maximo_de_dias_de_compra
FROM(
    SELECT
        id_cliente,
        COUNT(DISTINCT ano_mes_dia) AS freq
    FROM vendas
    WHERE faturamento > 0
    GROUP BY id_cliente
)
""").df()

,media_dos_dias_de_compra,mediana_dos_dias_de_compra,maximo_de_dias_de_compra
0,24.07797,7.0,482


In [26]:
# quantos clientes compraram somente uma vez?

con.execute("""
SELECT
    COUNT(*) AS clientes_com_1_compra
FROM (
    SELECT
        id_cliente,
        COUNT(DISTINCT ano_mes_dia) AS freq
    FROM vendas
    WHERE faturamento > 0
    GROUP BY id_cliente
)
WHERE freq = 1
""").df()

,clientes_com_1_compra
0,2841


In [27]:
# A grande diferença entre média e mediana indica que existem 2 grupos diferentes: clientes ocasionais e recorrentes.

# 2841 clientes compraram apenas 1 vez. Isso representa 24% dos 11684 clientes únicos no período analisado.

In [32]:
# além de medir a frequência de compra dos clientes, é preciso ver a recência deles ao final do período.

con.execute("""
    WITH ultima_compra AS (
        SELECT
            id_cliente,
            MAX(ano_mes_dia) AS data_ultima_compra
        FROM vendas
        WHERE faturamento > 0
        GROUP BY id_cliente
    )
    SELECT
        MIN(data_ultima_compra),
        MAX(data_ultima_compra)
    FROM ultima_compra
""").df()

,min(data_ultima_compra),max(data_ultima_compra)
0,20250102,20260531


In [41]:
# muitos clientes ativos até recentemente ou pararam de fazer compras nesta rede há mais tempo?

con.execute("""
    WITH ultima_compra AS (
        SELECT
            id_cliente,
            MAX(ano_mes_dia) AS data_ultima_compra
        FROM vendas
        WHERE faturamento > 0
        GROUP BY id_cliente
    )
    SELECT
        data_ultima_compra,
        COUNT(*) AS qtd_clientes
    FROM ultima_compra
    GROUP BY data_ultima_compra
    ORDER BY data_ultima_compra DESC
    LIMIT 20
""").df()

,data_ultima_compra,qtd_clientes
0,20260531,428
1,20260530,597
2,20260529,696
3,20260528,282
4,20260527,250
5,20260526,244
6,20260525,174
7,20260524,142
8,20260523,211
9,20260522,161


In [44]:
# Muitos clientes continuam ativos até o final da base (histórico recente)

# Sabendo disso, vou definir o período de 20250102 até 20260228 para observar o comportamento dos clientes.
# O período de 20260301 até 20260531 será usado para ver quais clientes abandonaram (churn = 1) e quais continuaram comprando (churn = 0).

# Com essa definição, qual a quantidade de clientes que abandonaram?

con.execute("""
    WITH clientes_observados AS (
        SELECT DISTINCT id_cliente
        FROM vendas
        WHERE ano_mes_dia <= 20260228
            AND faturamento > 0
    ),
            
    clientes_futuro AS (
        SELECT DISTINCT id_cliente
        FROM vendas
        WHERE ano_mes_dia BETWEEN 20260301 AND 20260531
            AND faturamento > 0
    )
    
    SELECT
        CASE
            WHEN f.id_cliente IS NULL THEN 'churn'
            ELSE 'ativo'
        END AS status,
        COUNT(*) AS qtd
    FROM clientes_observados o
    LEFT JOIN clientes_futuro f
        ON o.id_cliente = f.id_cliente
    GROUP BY status
""").df()

,status,qtd
0,ativo,5941
1,churn,4946


## Etapa 1

### Base RFM
-Recência

-Frequência

-Monetário (valor total)